# Approche 2 — `GlobalAggregationMessagePassingFunction` (Item 2)

Deuxième notebook d'attention pour l'équipe R&D RTE. Il déroule de bout en bout la `GlobalAggregationMessagePassingFunction` (un MLP de valeur par adresse, moyenné avec un dénominateur corrigé, puis rediffusé) sur le serveur GPU de La Javaness, dans le même pipeline `RecurrentCoupler` que l'Approche 1, sur les deux substrats.

**Prérequis :**
- Dépôt cloné sur le serveur GPU de La Javaness dans `./energnn-attention/`.
- `uv sync --extra gpu`, `pypowsybl==1.15.0` et `pypowsybl-to-energnn` installés.
- Branche `main` active (Approche 2 mergée).
- Notebooks 00 et 01 lus au préalable pour disposer des chiffres baseline et Approche 1.

**Durée attendue :** 7 à 12 minutes au total sur le serveur GPU de La Javaness.

**Structure en deux parties :**
- **Partie 1** — LinearSystem : GlobalAggregation vs LocalSum sur le toy DC.
- **Partie 2** — ieee9 + ieee14 : GlobalAggregation vs LocalSum sur AC load flow, avec Gate 6 (perf) et Gate 7 (reproductibilité).


## 1. Mise en place de l'environnement

Imports : JAX, NumPy, Flax NNX, optax. Vérification du device JAX et du dtype par défaut. On attend `CudaDevice` (serveur GPU de La Javaness) et `float32`.


In [1]:
import sys
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx

_root = Path.cwd().resolve()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent
sys.path.insert(0, str(_root / "benchmarks"))

print(f"JAX version:    {jax.__version__}")
print(f"JAX devices:    {jax.devices()}")
print(f"Default dtype:  {jnp.array(0.0).dtype}")
print(f"Repo root:      {_root}")


JAX version:    0.9.0


JAX devices:    [CudaDevice(id=0), CudaDevice(id=1)]


Default dtype:  float32
Repo root:      /mnt/nasbkp01/GPUDATA/van.tuan/energnn_attention_git


## 2. Méthode — `GlobalAggregationMessagePassingFunction`

La section 3.2 du backlog pose la formulation :

$$\psi_\theta(h, x)_a = \frac{1}{|\mathcal{A}_x|} \sum_{a' \in \mathcal{A}_x} \psi_\theta(h_{a'})$$

Chaque adresse receveuse reçoit le même résumé global des coordonnées. Enchâssée dans un `RecurrentCoupler`, elle donne à chaque adresse une vue globale du graphe. L'Approche 5 (`VirtualAddressRecurrentCoupler`) réutilise la même moyenne masquée + dénominateur corrigé dans `F_virtual`, mais sans appeler `GlobalAggregationMessagePassingFunction` (l'Item 5 v1 enveloppe LocalSum).

Deux différences mécaniques avec `LocalSumMessagePassingFunction` :

1. **Pas de factorisation par couple `(classe, port)`** : la formulation ne fait référence qu'à `h_{a'}` (coordonnée à l'adresse `a'`), sans classe ni port. L'implémentation utilise un unique `value_mlp` appliqué directement aux coordonnées.

2. **L'agrégation est une moyenne sur toutes les adresses**, pas une somme sur les voisins par couple `(classe, port)`. Le résultat est rediffusé à tous les receveurs, masqué sur les fictives.

**Préoccupation scientifique n°1 du backlog (§3.2)** : le dénominateur littéral $|\mathcal{A}_x|$ inclut le padding fictif et dilue le signal. L'implémentation applique un **dénominateur corrigé** :

$$\psi_\theta(h, x)_a = \sigma\!\left( \frac{1}{|\mathcal{A}_x^{\mathrm{real}}| + \varepsilon} \sum_{a' \in \mathcal{A}_x^{\mathrm{real}}} \xi_\theta(h_{a'}) \right)$$

où $\mathcal{A}_x^{\mathrm{real}}$ est l'ensemble des adresses non fictives, $\varepsilon = 10^{-9}$ protège le cas dégénéré d'ensemble réel vide, et $\sigma$ est l'`outer_activation`.

**Périmètre v1** (suivant la *proposed resolution* du backlog) : moyenne uniquement. Les formes multi-tête, attention pondérée et les reducers min/max/sum sont reportés à un ticket v2.

**Propriété unique de GlobalAggregation** : permutation **invariante** (plus fort que l'équivariance) — la moyenne est symétrique en ses arguments, la valeur rediffusée est indépendante de toute permutation des adresses.


# Partie 1 — Expériences sur LinearSystem

GlobalAggregation vs LocalSum sur le DC power flow toy. Substrat minimal pour valider le pipeline et la propriété de broadcast (même résumé global pour toutes les adresses). La mesure de capacité se fait en partie 2 sur AC LF.


## 3.1. Construction de `LinearSystemProblemLoader` et helper

Loader identique à ceux des Approches 0 et 1 (même seed, même configuration), afin d'aligner les valeurs d'évaluation. Le helper `build_gnn` paramètre la message function — on lui passe `LocalSumMessagePassingFunction` ou `GlobalAggregationMessagePassingFunction` pour comparer sur le même pipeline.


In [2]:
from energnn.model import (
    GNN, MLP, MLPEncoder, MLPEquivariantDecoder, RecurrentCoupler,
    GlobalAggregationMessagePassingFunction, LocalSumMessagePassingFunction,
    TDigestNormalizer,
)
from energnn.problem.example import LinearSystemProblemLoader
from energnn.trainer import Trainer

LATENT_DIM = 4  # Tiny config

ls_train = LinearSystemProblemLoader(seed=0)
ls_val = LinearSystemProblemLoader(seed=42)


def build_gnn(in_struct, out_struct, *, message_fn_cls, rngs):
    """Build a Tiny GNN whose message function is `message_fn_cls`.

    `message_fn_cls` is either LocalSumMessagePassingFunction or
    GlobalAggregationMessagePassingFunction. Both share the relevant subset of
    constructor surface (in_graph_structure, in_array_size, hidden_sizes,
    activation, out_size, use_bias, final_activation, outer_activation, rngs).
    LocalSum's additional encoded_feature_size is accepted via kwargs filter.
    """
    msg_kwargs = dict(
        in_graph_structure=in_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
        activation=nnx.leaky_relu, out_size=LATENT_DIM, use_bias=True,
        final_activation=None, outer_activation=nnx.tanh, rngs=rngs,
    )
    if message_fn_cls is LocalSumMessagePassingFunction:
        msg_kwargs["encoded_feature_size"] = LATENT_DIM
    msg_fn = message_fn_cls(**msg_kwargs)
    return GNN(
        normalizer=TDigestNormalizer(in_structure=in_struct, n_breakpoints=10, update_limit=1000),
        encoder=MLPEncoder(
            in_structure=in_struct, hidden_sizes=[], activation=nnx.leaky_relu,
            out_size=LATENT_DIM, use_bias=True, final_activation=None, rngs=rngs,
        ),
        coupler=RecurrentCoupler(
            phi=MLP(
                in_size=LATENT_DIM, hidden_sizes=[], activation=nnx.leaky_relu,
                out_size=LATENT_DIM, use_bias=True, final_activation=nnx.tanh, rngs=rngs,
            ),
            message_functions=[msg_fn],
            n_steps=4,
        ),
        decoder=MLPEquivariantDecoder(
            in_graph_structure=in_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
            activation=nnx.leaky_relu, out_structure=out_struct, use_bias=True,
            final_activation=None, encoded_feature_size=LATENT_DIM, rngs=rngs,
        ),
    )


def count_params(model):
    _, params, _ = nnx.split(model, nnx.Param, ...)
    return sum(int(leaf.size) for leaf in jax.tree_util.tree_leaves(params) if hasattr(leaf, "size"))


print(f"train loader: {type(ls_train).__name__}, seed=0")
print(f"val loader:   {type(ls_val).__name__}, seed=42")
print(f"build_gnn(...) helper ready")


train loader: LinearSystemProblemLoader, seed=0
val loader:   LinearSystemProblemLoader, seed=42
build_gnn(...) helper ready


## 3.2. Vérification de la propriété de broadcast et du dénominateur corrigé

Deux propriétés propres à GlobalAggregation :

1. **Invariance par broadcast** : les lignes de sortie sont identiques sur les receveurs réels (chaque receveur reçoit le même résumé global). L'équivariance de LocalSum / GATv2 demande seulement que la sortie permute avec l'entrée ; l'invariance de GlobalAgg est plus forte — les valeurs sont égales quelle que soit la permutation.

2. **Dénominateur corrigé** : le spec littéral utilise $|\mathcal{A}_x|$ y compris le padding ; l'implémentation utilise $|\mathcal{A}_x^{\mathrm{real}}| + \varepsilon$ (préoccupation scientifique n°1 du backlog).

La cellule suivante vérifie les deux propriétés à l'aide d'un MLP identité patché, ce qui rend la moyenne vérifiable bit-exact analytiquement.


In [3]:
rngs_test = nnx.Rngs(0)
mf_test = GlobalAggregationMessagePassingFunction(
    in_graph_structure=ls_train.context_structure,
    in_array_size=LATENT_DIM, hidden_sizes=[],
    activation=nnx.leaky_relu, out_size=LATENT_DIM,
    use_bias=False, final_activation=None,
    outer_activation=nnx.identity, rngs=rngs_test,
)
# Patch the single Linear weight with identity so value_mlp(coord) == coord.
linear = mf_test.value_mlp.sequential.layers[0]
linear.kernel.value = jnp.eye(LATENT_DIM, dtype=jnp.float32)

ls_batch = next(iter(ls_train))
ls_ctx_batch, _ = ls_batch.get_context()
ls_ctx = jax.tree.map(lambda x: x[0], ls_ctx_batch)
n_addr_ls = int(ls_ctx.non_fictitious_addresses.shape[0])
n_real_ls = int(ls_ctx.non_fictitious_addresses.sum())

coords_test = jnp.array(np.random.default_rng(7).normal(size=(n_addr_ls, LATENT_DIM)).astype(np.float32))
out_test, _ = mf_test(graph=ls_ctx, coordinates=coords_test, get_info=False)

# (1) Broadcast: every real receiver row identical.
mask = np.asarray(ls_ctx.non_fictitious_addresses).astype(bool)
real_rows = np.asarray(out_test)[mask]
max_row_diff = float(np.max(np.abs(real_rows - real_rows[0]))) if real_rows.shape[0] >= 2 else 0.0
print(f"Broadcast property:")
print(f"  n_addr={n_addr_ls}, n_real={n_real_ls}")
print(f"  max |row_i - row_0| on real receivers: {max_row_diff:.2e}")
assert max_row_diff == 0.0, "broadcast broken"

# (2) Corrected denominator: identity MLP, mean over REAL addresses only.
expected_mean = jnp.sum(coords_test * jnp.expand_dims(ls_ctx.non_fictitious_addresses, -1), axis=0) / (
    jnp.sum(ls_ctx.non_fictitious_addresses) + mf_test.eps
)
first_real_idx = int(np.argmax(mask))
diff_mean = float(jnp.max(jnp.abs(out_test[first_real_idx] - expected_mean)))
print(f"\nMean correctness (corrected denominator):")
print(f"  expected mean (analytical): {np.asarray(expected_mean)[:4]}")
print(f"  observed mean (impl):       {np.asarray(out_test[first_real_idx])[:4]}")
print(f"  max |diff|: {diff_mean:.2e}")
assert diff_mean < 1e-6

print("\nPASS: broadcast invariance + corrected denominator")


Broadcast property:
  n_addr=4, n_real=2
  max |row_i - row_0| on real receivers: 0.00e+00



Mean correctness (corrected denominator):
  expected mean (analytical): [-0.22672032 -0.3464505  -0.10699712  0.22481167]
  observed mean (impl):       [-0.22672032 -0.3464505  -0.10699712  0.22481167]
  max |diff|: 0.00e+00

PASS: broadcast invariance + corrected denominator


## 3.3. Entraînement de GlobalAggregation vs LocalSum sur LinearSystem (5 epochs)

Deux GNN identiques à la message function près, entraînés sur les mêmes données, le même seed et la même configuration, évalués après chaque epoch.

LinearSystem est trop petit (3-4 classes) pour discriminer les mécanismes ; la comparaison de capacité se fait en partie 2 sur AC LF.


In [4]:
def train_and_eval(gnn, train_loader, val_loader, n_epochs):
    trainer = Trainer(model=gnn, gradient_transformation=optax.adam(1e-3))
    init, _ = trainer.eval(val_loader, progress_bar=False)
    curve = [float(init)]
    for _ in range(n_epochs):
        trainer.train(train_loader=train_loader, val_loader=None, n_epochs=1,
                      progress_bar=False, eval_before_training=False, eval_after_epoch=False)
        s, _ = trainer.eval(val_loader, progress_bar=False)
        curve.append(float(s))
    return curve


N_EPOCHS_LS = 5
ls_curves = {}
for label, mf_cls in (("LocalSum",          LocalSumMessagePassingFunction),
                     ("GlobalAggregation", GlobalAggregationMessagePassingFunction)):
    gnn = build_gnn(ls_train.context_structure, ls_train.decision_structure,
                    message_fn_cls=mf_cls, rngs=nnx.Rngs(0))
    n_params = count_params(gnn)
    ls_curves[label] = train_and_eval(gnn, ls_train, ls_val, N_EPOCHS_LS)
    print(f"{label:<20s} n_params={n_params:>5d}  init={ls_curves[label][0]:.3e}  final={ls_curves[label][-1]:.3e}")

print(f"\n{'epoch':<8s} {'LocalSum':>14s} {'GlobalAgg':>14s} {'Δ vs LocalSum':>16s}")
print("-" * 60)
for ep, (l, g) in enumerate(zip(ls_curves["LocalSum"], ls_curves["GlobalAggregation"])):
    label = "init" if ep == 0 else f"ep {ep}"
    delta = 100.0 * (g - l) / l if l > 0 else 0.0
    print(f"{label:<8s} {l:>14.4e} {g:>14.4e} {delta:>14.1f}%")


LocalSum             n_params=  185  init=4.981e-01  final=3.925e-01


GlobalAggregation    n_params=   65  init=6.982e-01  final=6.607e-01

epoch          LocalSum      GlobalAgg    Δ vs LocalSum
------------------------------------------------------------
init         4.9811e-01     6.9821e-01           40.2%
ep 1         4.7210e-01     6.9082e-01           46.3%
ep 2         4.4839e-01     6.8325e-01           52.4%
ep 3         4.2734e-01     6.7572e-01           58.1%
ep 4         4.0902e-01     6.6822e-01           63.4%
ep 5         3.9254e-01     6.6072e-01           68.3%


# Partie 2 — Expériences sur IEEE-9 et IEEE-14 (AC load flow)

Substrat de capacité : 11 classes d'hyper-arêtes, AC LF non linéaire, oracle V_mag / P / Q / I issu de pypowsybl. Le notebook se limite à 3 epochs Tiny ; le Gate 5 complet (Tiny + Small × 3 seeds × 15 epochs) est consigné dans `benchmarks/results/02_global_aggregation/baseline_global_aggregation_ac_load_flow.json`.


## 4.1. Construction de `ACLoadFlowProblemLoader` pour ieee9 et ieee14

Loader pre-cache (commit `7d5f9f8`) : 32 instances générées à `__init__`, réutilisées à chaque epoch. Même seed que les baselines des notebooks 00 et 01, ce qui retire le drift float du training data des deltas entre Approches.


In [5]:
import time
from ac_load_flow_problem import ACLoadFlowProblemLoader

LOADERS = {}
for net in ("ieee9", "ieee14"):
    t0 = time.perf_counter()
    LOADERS[net] = {
        "train": ACLoadFlowProblemLoader(network_name=net, dataset_size=32, batch_size=4,
                                          seed=0, perturbation_scale=0.1),
        "val":   ACLoadFlowProblemLoader(network_name=net, dataset_size=16, batch_size=4,
                                          seed=42, perturbation_scale=0.1),
    }
    print(f"{net}: built train+val loaders in {time.perf_counter() - t0:.1f}s")


ieee9: built train+val loaders in 6.9s


ieee14: built train+val loaders in 6.5s


## 4.2. Entraînement de GlobalAggregation vs LocalSum sur ieee9 et ieee14 (3 epochs)

3 epochs Tiny par couple (réseau, message_fn). Sanity-check du pipeline GlobalAggregation sur AC LF ; chiffres canoniques (3 seeds × Tiny/Small × 10-15 epochs) dans `BASELINES.md` section Approche 2.


In [6]:
N_EPOCHS_IEEE = 3
ieee_results = {}

for net in ("ieee9", "ieee14"):
    train_loader = LOADERS[net]["train"]
    val_loader = LOADERS[net]["val"]
    ieee_results[net] = {}
    for label, mf_cls in (("LocalSum",          LocalSumMessagePassingFunction),
                          ("GlobalAggregation", GlobalAggregationMessagePassingFunction)):
        gnn = build_gnn(train_loader.context_structure, train_loader.decision_structure,
                        message_fn_cls=mf_cls, rngs=nnx.Rngs(0))
        t0 = time.perf_counter()
        curve = train_and_eval(gnn, train_loader, val_loader, N_EPOCHS_IEEE)
        ieee_results[net][label] = {
            "curve": curve, "n_params": count_params(gnn),
            "train_time_s": time.perf_counter() - t0,
        }

print(f"{'network':<8s} {'method':<22s} {'n_params':>9s} {'init':>12s} {'final':>12s} {'train (s)':>11s}")
print("-" * 80)
for net in ("ieee9", "ieee14"):
    for label in ("LocalSum", "GlobalAggregation"):
        r = ieee_results[net][label]
        print(f"{net:<8s} {label:<22s} {r['n_params']:>9d} {r['curve'][0]:>12.3e} "
              f"{r['curve'][-1]:>12.3e} {r['train_time_s']:>11.1f}")

print(f"\n{'network':<8s} {'LocalSum final':>16s} {'GlobalAgg final':>17s} {'Δ vs LocalSum':>16s}")
print("-" * 65)
for net in ("ieee9", "ieee14"):
    l = ieee_results[net]["LocalSum"]["curve"][-1]
    g = ieee_results[net]["GlobalAggregation"]["curve"][-1]
    delta = 100.0 * (g - l) / l if l > 0 else 0.0
    print(f"{net:<8s} {l:>16.4e} {g:>17.4e} {delta:>14.1f}%")


network  method                  n_params         init        final   train (s)
--------------------------------------------------------------------------------
ieee9    LocalSum                    1587    7.012e-01    5.140e-01       101.1
ieee9    GlobalAggregation            719    5.556e-01    4.833e-01        74.7
ieee14   LocalSum                    1587    3.817e-01    2.799e-01       129.5
ieee14   GlobalAggregation            719    3.234e-01    2.636e-01       103.4

network    LocalSum final   GlobalAgg final    Δ vs LocalSum
-----------------------------------------------------------------
ieee9          5.1402e-01        4.8330e-01           -6.0%
ieee14         2.7994e-01        2.6356e-01           -5.9%


## 4.3. Gate 6 — surcoût GlobalAggregation vs LocalSum

Mesure du temps médian forward et forward + backward des deux message functions isolées sur le context ieee14. Mêmes hyper-paramètres, code JIT-compilé, 20 itérations de warm-up puis 100 itérations chronométrées.

GlobalAggregation est mécaniquement plus léger que LocalSum (un seul MLP + 1 somme + 1 broadcast contre l'arbre MLP par couple `(classe, port)` + un `scatter_add` par port de LocalSum). Les chiffres Gate 6 complets (LinearSystem + ieee118 + ieee300) sont dans `benchmarks/results/02_global_aggregation/perf_global_aggregation_vs_localsum.json`.

In [7]:
import statistics

ieee14_loader = LOADERS["ieee14"]["train"]
ctx14, _ = ieee14_loader._cached_pairs[0]
n_addr14 = int(ctx14.non_fictitious_addresses.shape[0])
perf_coords = jnp.full((n_addr14, 4), 0.5, dtype=jnp.float32)

mf_perf_ls = LocalSumMessagePassingFunction(
    in_graph_structure=ieee14_loader.context_structure, in_array_size=4, out_size=4,
    hidden_sizes=[4], activation=nnx.leaky_relu, final_activation=None,
    outer_activation=nnx.tanh, seed=64,
)
mf_perf_ga = GlobalAggregationMessagePassingFunction(
    in_graph_structure=ieee14_loader.context_structure, in_array_size=4, out_size=4,
    hidden_sizes=[4], activation=nnx.leaky_relu, final_activation=None,
    outer_activation=nnx.tanh, seed=64,
)


def make_jit_funcs(mf):
    @nnx.jit
    def fwd(m, c):
        out, _ = m(graph=ctx14, coordinates=c, get_info=False)
        return out

    @nnx.jit
    def fb(m, c):
        def loss(mod):
            out, _ = mod(graph=ctx14, coordinates=c, get_info=False)
            return jnp.mean(out ** 2)
        return nnx.value_and_grad(loss)(m)
    return fwd, fb


def time_calls(callable_, n_warmup=20, n_measure=100):
    for _ in range(n_warmup):
        out = callable_(); jax.block_until_ready(out)
    timings = []
    for _ in range(n_measure):
        t0 = time.perf_counter()
        out = callable_(); jax.block_until_ready(out)
        timings.append(time.perf_counter() - t0)
    return statistics.median(timings) * 1000.0


fwd_ls, fb_ls = make_jit_funcs(mf_perf_ls)
fwd_ga, fb_ga = make_jit_funcs(mf_perf_ga)

ls_fwd_ms = time_calls(lambda: fwd_ls(mf_perf_ls, perf_coords))
ga_fwd_ms = time_calls(lambda: fwd_ga(mf_perf_ga, perf_coords))
ls_fb_ms = time_calls(lambda: fb_ls(mf_perf_ls, perf_coords))
ga_fb_ms = time_calls(lambda: fb_ga(mf_perf_ga, perf_coords))

print(f"{'':18s} {'LocalSum':>10s} {'GlobalAgg':>12s} {'overhead':>10s}")
print("-" * 56)
print(f"{'forward (ms)':18s} {ls_fwd_ms:>10.3f} {ga_fwd_ms:>12.3f} {ga_fwd_ms / ls_fwd_ms:>9.2f}x")
print(f"{'fwd+bwd (ms)':18s} {ls_fb_ms:>10.3f} {ga_fb_ms:>12.3f} {ga_fb_ms / ls_fb_ms:>9.2f}x")


                     LocalSum    GlobalAgg   overhead
--------------------------------------------------------
forward (ms)            8.304        0.847      0.10x
fwd+bwd (ms)           10.314        1.054      0.10x


## 4.4. Gate 7 — hash de reproductibilité in-process

Avec un seed fixé et un context IEEE-14 figé (chargé depuis le cache pickle `benchmarks/results/02_global_aggregation/consistency_global_aggregation_context.pkl`), la sortie forward de GlobalAggregation doit produire la même empreinte numérique entre re-runs in-process. Toute différence signalerait une régression de déterminisme.

Le Gate 7 officiel isole la reproductibilité GlobalAggregation via un cache de context pickle exécuté dans le script autonome `benchmarks/02_global_aggregation/consistency_global_aggregation.py`, le hash de référence étant écrit dans `benchmarks/results/02_global_aggregation/consistency_global_aggregation.json`. La bit-identité cross-process n'est vérifiable qu'avec ce script, pas via le notebook (cache JIT et ordonnancement des kernels GPU diffèrent).


In [8]:
import hashlib

# Run the standalone consistency script once to ensure the context cache exists.
# Then load the same cache and verify in-process forward determinism.
cache_path = _root / "benchmarks/results/02_global_aggregation/consistency_global_aggregation_context.pkl"
if not cache_path.exists():
    # Trigger consistency script which builds the cache.
    import subprocess
    print("Building consistency context cache (first run only)...")
    subprocess.run(
        ["python", str(_root / "benchmarks/02_global_aggregation/consistency_global_aggregation.py")],
        cwd=str(_root), check=True,
    )

import pickle
with cache_path.open("rb") as f:
    blob = pickle.load(f)
fixed_context = blob["context"]
fixed_structure = blob["structure"]

mf_fixed = GlobalAggregationMessagePassingFunction(
    in_graph_structure=fixed_structure, in_array_size=4, out_size=4,
    hidden_sizes=[4], activation=nnx.leaky_relu, final_activation=None,
    outer_activation=nnx.tanh, seed=64,
)
n_addr_fixed = int(fixed_context.non_fictitious_addresses.shape[0])
fixed_coords = jnp.full((n_addr_fixed, 4), 0.5, dtype=jnp.float32)


def _forward_hash():
    out, _ = mf_fixed(graph=fixed_context, coordinates=fixed_coords, get_info=False)
    return hashlib.sha256(np.asarray(out, dtype=np.float64).tobytes()).hexdigest()


hash_a = _forward_hash()
hash_b = _forward_hash()

print(f"observed hash, call 1: {hash_a}")
print(f"observed hash, call 2: {hash_b}")
print(f"in-process match:      {hash_a == hash_b}")
assert hash_a == hash_b, "forward not in-process deterministic"
print("\nPASS: forward in-process deterministic (same hash across calls within same kernel).")
print("Cross-process bit-identical reference verified by")
print("  uv run python benchmarks/02_global_aggregation/consistency_global_aggregation.py")
print("(reference hash in benchmarks/results/02_global_aggregation/consistency_global_aggregation.json).")


observed hash, call 1: 6c2d2c1a0a58ec5476b243c0b46713284bf123aa4bceaf7b00bae011eec45c99
observed hash, call 2: 6c2d2c1a0a58ec5476b243c0b46713284bf123aa4bceaf7b00bae011eec45c99
in-process match:      True

PASS: forward in-process deterministic (same hash across calls within same kernel).
Cross-process bit-identical reference verified by
  uv run python benchmarks/02_global_aggregation/consistency_global_aggregation.py
(reference hash in benchmarks/results/02_global_aggregation/consistency_global_aggregation.json).


## 5. Synthèse et références

Résumé des observations de ce notebook (Approche 2, Item 2 GlobalAggregation) :

- **LinearSystem** : LocalSum et GlobalAggregation convergent tous deux sur le toy DC.
- **ieee9 / ieee14 Tiny (3 epochs)** : le pipeline GlobalAgg tourne, l'eval décroît régulièrement, aucun NaN.
- **Gate 6** : GlobalAggregation est mécaniquement plus léger que LocalSum (1 MLP contre un arbre MLP). Sur ieee14, le ratio forward et fwd+bwd mesuré est de 0.10x (environ 10x plus rapide).
- **Gate 7** : déterminisme forward in-process vérifié ; la bit-identité complète passe par le script autonome.

Les résultats complets de l'Approche 2 (3 seeds × Tiny/Small × 10/15 epochs) sont consignés dans :

- `BASELINES.md` — chiffres Gate 5 + comparaison côte-à-côte LocalSum vs GlobalAgg + observations coût vs capacité.
- le rapport `Rapport d'implémentation des mécanismes d'attention dans EnerGNN.pdf`, chapitre 12 — analyse technique détaillée, formulation, dérivation du dénominateur corrigé, caractérisation perf.
- `benchmarks/results/02_global_aggregation/*.json` — JSON bruts Gate 5/6/7.